In [ ]:
import geopandas as gpd
from shapely.ops import unary_union 
import unicodedata

In [ ]:

c_2010 = gpd.read_file(r"D:\Users\ivan.cavalcanti\Documents\Projects\areas-verdes\data\soil_use_2010.shp")
c_2023 = gpd.read_file(r"D:\Users\ivan.cavalcanti\Documents\Projects\areas-verdes\data\soil_use_2023.shp")

In [ ]:
b = gpd.read_file(r"D:\Users\ivan.cavalcanti\Documents\Projects\areas-verdes\data\Geojundiai\L_2405-1980_Manancial.shp")

In [ ]:
c = gpd.read_file(r"D:\Users\ivan.cavalcanti\Documents\Projects\areas-verdes\data\Geojundiai\L_7858-2012-zoneamento.shp")

In [ ]:
a = c[c['l7858_macr'].isin(['Reserva Biológica', 'Zona de Conservacao Rural',
       'Território de Gestão da Serra do Japi'])]

In [ ]:
a

In [ ]:
z = gpd.read_file(r"D:\Arq-Azzoni\UrbanSprawl\Bases_dados\Shapes\jundiai\zoneamento_jundiai_20210118\zoneamento_jundiai_20210118.shp")

In [ ]:
z

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx

gdf = z

# Reproject to Web Mercator (required for basemap tiles)
gdf = gdf.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(10, 10))
gdf.plot(ax=ax, edgecolor="black", facecolor="lightblue", alpha=0.6)

cx.add_basemap(ax, source=cx.providers.Esri.WorldImagery)

ax.set_axis_off()
plt.show()

In [ ]:
import geopandas as gpd



# SIRGAS 2000 / UTM 23S — correct for São Paulo region
zones = zones.to_crs(epsg=31983)
protected = protected.to_crs(zones.crs)

# Dissolve protected areas to avoid double-counting overlaps
protected = protected.dissolve()

# Total area of each zone (km²)
zones["total_area"] = zones.geometry.area / 1_000_000

# Intersect zones with protected areas, sum protected area per zone
overlay = gpd.overlay(zones, protected, how="intersection")
overlay["prot_area"] = overlay.geometry.area / 1_000_000
prot_by_zone = overlay.groupby("CD_ZONA")["prot_area"].sum()

# Map back to zones (0 where no overlap)
zones["prot_area"] = zones["CD_ZONA"].map(prot_by_zone).fillna(0)

# Usable area (km²)
zones["usable_area"] = zones["total_area"] - zones["prot_area"]

zones.to_file("zones_with_area.shp")

In [ ]:
import geopandas as gpd

zones = z
protected = a
# Use a projected CRS in meters for correct area calculation
# (replace with the appropriate UTM/local CRS for your city)
zones = zones.to_crs(epsg=31983)
protected = protected.to_crs(zones.crs)

# Total area of each zone (m²)
zones["total_area"] = zones.geometry.area

# Intersect zones with protected areas, then sum protected area per zone
overlay = gpd.overlay(zones, protected, how="intersection")
overlay["prot_area"] = overlay.geometry.area

prot_by_zone = overlay.groupby("CD_ZONA")["prot_area"].sum()

# Map protected area back to zones (0 where no overlap)
zones["prot_area"] = zones["CD_ZONA"].map(prot_by_zone).fillna(0)

# Usable area = total - protected
zones["usable_area"] = zones["total_area"] - zones["prot_area"]

zones.to_file("zonas_com_area.shp")

In [ ]:
zones

In [ ]:


# --- helpers rápidos ---
def normalize_txt(s: str) -> str:
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode("ascii")
    return s.lower().strip()

def filtro_urbano(g: gpd.GeoDataFrame, coluna_classe="soil_use") -> gpd.GeoDataFrame:
    if coluna_classe not in g.columns:
        raise ValueError(f"Coluna de classe '{coluna_classe}' não encontrada")
    g = g.copy()
    g[coluna_classe] = g[coluna_classe].map(normalize_txt)
    return g[g[coluna_classe].str.contains(r"\burbano\b")].copy()

def fix_valid(g: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    try:
        from shapely import make_valid
        g["geometry"] = g.geometry.apply(make_valid)
    except Exception:
        g["geometry"] = g.buffer(0)
    return g

def to_same_crs(g1: gpd.GeoDataFrame, g2: gpd.GeoDataFrame) -> tuple[gpd.GeoDataFrame, gpd.GeoDataFrame]:
    if g1.crs is None or g2.crs is None:
        raise ValueError("Algum layer está sem CRS; defina antes de continuar.")
    if g1.crs != g2.crs:
        g2 = g2.to_crs(g1.crs)
    return g1, g2

def diff_ano(g_new_u: gpd.GeoDataFrame, g_old_u: gpd.GeoDataFrame, ano_new: int, ano_old: int) -> gpd.GeoDataFrame:
    g_new_u = fix_valid(g_new_u)
    g_old_u = fix_valid(g_old_u)
    g_new_u, g_old_u = to_same_crs(g_new_u, g_old_u)

    new_u = unary_union(g_new_u.geometry)
    old_u = unary_union(g_old_u.geometry)
    d = new_u.difference(old_u)

    if d.is_empty:
        return gpd.GeoDataFrame(columns=["year_from", "year_to", "geometry"], crs=g_new_u.crs)

    gdiff = gpd.GeoDataFrame(geometry=[d], crs=g_new_u.crs).explode(index_parts=False, ignore_index=True)

    # (opcional) remover polígonos muito pequenos — reprojeta para métrico rápido
    try:
        g_m = gdiff.to_crs(3857)  # metros (aprox.)
        keep = g_m.area > 500     # ex.: > 500 m²; ajuste conforme necessário
        gdiff = gdiff[keep].copy()
    except Exception:
        pass

    # volta para WGS84 para o Folium (se quiser)
    if gdiff.crs.to_epsg() != 4326:
        gdiff = gdiff.to_crs(4326)

    gdiff["year_from"] = int(ano_old)
    gdiff["year_to"]   = int(ano_new)
    return gdiff

# --- uso ---
# Se você já tem gdf_2010 e gdf_2023 lidos:
gdf_2010 = c_2010
gdf_2023 = c_2023

g2010_u = filtro_urbano(gdf_2010, coluna_classe="soil_use")  # ajuste o nome da coluna se for outro
g2023_u = filtro_urbano(gdf_2023, coluna_classe="soil_use")

gdiff_10_23 = diff_ano(g_new_u=g2023_u, g_old_u=g2010_u, ano_new=2023, ano_old=2010)

print(gdiff_10_23.head())

In [ ]:
c_2010.head()

In [ ]:
path = r"D:\Arq-Azzoni\UrbanSprawl\Bases_dados\Shapes\area_ponderacao_2023_com_mun\area_ponderacao_2023_com_mun.shp"

# ler o shapefile
gdf = gpd.read_file(path)

In [ ]:
contagem = gdf.groupby("NM_MUN").size().reset_index(name="n_linhas")

In [ ]:
contagem.sort_values("n_linhas", ascending=False)

In [ ]:


# Filtrar municípios com apenas 1 linha
municipios_1 = contagem[contagem["n_linhas"] == 1]

In [ ]:
municipios_1

In [ ]:
gdf